In [ ]:
# DFU Phase-4 Universal Safe V6 — Drive-first Colab, Kaggle auto-fallback, CPU/GPU auto, resume-safe
import urllib.request, hashlib, base64, zlib

VERSION = "DFU_PHASE4_UNIVERSAL_SAFE_V6_LOADER_20260812"
SOURCE_COMMIT = "8237051190de164d450db676ad7eadab563755c5"
PARTS = [
    ("scripts/phase4_universal_safe_v6_payload/part_00.txt", "0d519e8c668224621c82adab2b16daf7703fd93e"),
    ("scripts/phase4_universal_safe_v6_payload/part_01.txt", "0aee060330a2f82c1ecc1997dc3c8aca201b564e"),
    ("scripts/phase4_universal_safe_v6_payload/part_02.txt", "0e8284bb3ff5551be96a13a253b2b88042d04297"),
    ("scripts/phase4_universal_safe_v6_payload/part_03.txt", "25e28bcd38cc055b4711baaf55728a9581acca23"),
]
EXPECTED_PAYLOAD_SHA256 = "f78e5f98232e8b7373496efb7929ccbaa9fbb5b765c21b7783268c3dfa860641"
EXPECTED_SOURCE_SHA256 = "d62379dcdd5f6b790f6c45e1e4e54ac752e9cc583de890fa9c32381b1a104b6e"
BASE = f"https://raw.githubusercontent.com/AzizulHakim00/DFU-ImageGuard/{SOURCE_COMMIT}/"

def git_blob_sha(raw):
    return hashlib.sha1(b"blob " + str(len(raw)).encode() + b"\0" + raw).hexdigest()

print("="*100)
print(VERSION)
print("DRIVE-FIRST COLAB | KAGGLE AUTO-FALLBACK | CPU/GPU AUTO | RESUME-SAFE")
print("NO TRAINING | NO FINE-TUNING | NO EXTERNAL THRESHOLD/CALIBRATION FITTING")
print("="*100)

chunks=[]
for path, expected_blob in PARTS:
    raw=urllib.request.urlopen(BASE+path, timeout=120).read()
    actual_blob=git_blob_sha(raw)
    if actual_blob != expected_blob:
        raise RuntimeError(f"Payload fragment mismatch: {path} expected={expected_blob} actual={actual_blob}")
    print("Payload fragment PASS:", path, actual_blob)
    chunks.append(raw)

payload=b"".join(chunks)
payload_sha=hashlib.sha256(payload).hexdigest()
if payload_sha != EXPECTED_PAYLOAD_SHA256:
    raise RuntimeError(f"Payload SHA mismatch: expected={EXPECTED_PAYLOAD_SHA256} actual={payload_sha}")
print("Combined payload SHA256: PASS", payload_sha)

source=zlib.decompress(base64.b64decode(payload, validate=True))
source_sha=hashlib.sha256(source).hexdigest()
if source_sha != EXPECTED_SOURCE_SHA256:
    raise RuntimeError(f"Decoded source SHA mismatch: expected={EXPECTED_SOURCE_SHA256} actual={source_sha}")
print("Decoded source SHA256: PASS", source_sha)
text=source.decode("utf-8")
compile(text, "dfu_phase4_universal_safe_v6.py", "exec")
print("Decoded source compile: PASS")
print("Starting tested Phase-4 V6...")
exec(compile(text, "dfu_phase4_universal_safe_v6.py", "exec"), globals())
